<a href="https://colab.research.google.com/github/marioarsw/AnaliticaDescriptiva/blob/main/practicas/practica6_267370.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Maestría en Inteligencia Artificial y Analítica de Datos


*   Curso: Programación para Analítica Descriptiva y Predictiva
*   Semestre: Enero-Junio
*   Profesor: Dr. Vicente García Jiménez



## Práctica 6: Manejo de la librería Pandas

## Mario Amador - 267370.

Instrucciones: Carga el archivo titanic.csv en la carpeta correspondiente de Google drive para realizar los siguientes ejercicios:

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
titanic = sns.load_dataset("titanic")
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# Ejercicio 1: Análisis de la distribución de supervivencia por combinación de sexo y clase del pasajero.

*   Calcula la proporción de supervivencia para cada combinación de 'Sex' y 'Pclass'.
*    Identifica qué combinación tuvo la tasa de supervivencia más alta.
*   Identifica qué combinación tuvo la tasa de supervivencia más baja.

In [4]:
# Primero agrupar por sexo y clase para tener la siguiente forma:

# mujer - clase 1
# mujer - clase 2
# mujer - clase 3
# hombre - clase 1
# hombre - clase 2
# hombre - clase 3

titanic.groupby(['sex','pclass'])

In [18]:
# Calcular la proporción de supervivencia
survival_rate = titanic.groupby(['sex','pclass'])['survived'].mean()
# survival_rate

# O bien escrito de otra manera
# survival_rate = (
#     titanic
#     .groupby(['sex', 'pclass'])['survived']
#     .mean()
# )

# Convertirlo a df para poder localizar el max y el min
survival_rate = survival_rate.reset_index()
survival_rate


,sex,pclass,survived
0,female,1,0.968085
1,female,2,0.921053
2,female,3,0.500000
3,male,1,0.368852
4,male,2,0.157407
5,male,3,0.135447


Aunque en lo anterior ya es visible la tasa con mayor supervivencia y la que tiene menor supervivencia, también se puede encontrar los valroes con loc y mándandole la columna de survived.

In [19]:
survival_rate.loc[survival_rate['survived'].idxmax()]

,0
sex,female
pclass,1
survived,0.968085


In [20]:
survival_rate.loc[survival_rate['survived'].idxmin()]

,5
sex,male
pclass,3
survived,0.135447


# Ejercicio 2: Identificación de familias grandes a bordo.

* Crea una nueva columna 'FamilySize' sumando las columnas 'SibSp' y 'Parch'.
* Considera como "familia grande" a aquellas donde 'FamilySize' es mayor a 3.
* Calcula el número de pasajeros en familias grandes.
* Calcula la proporción de supervivencia entre los pasajeros que pertenecen a familias grandes.

In [21]:
# Crear nueva columna
# sibsp -> número de hermanos / cónyuge a bordo
# parch -> número de padres / hijos a bordo
# nota -> no se está contando al pasajero principal
titanic['family_size'] = titanic['sibsp'] + titanic['parch']
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,1
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,1
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,0
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,0
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,3
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,0


In [23]:
# definir familia grande en un nuevo df
large_family = titanic['family_size'] > 3
large_family
# A esto se le llama máscara booleana, como solo necesitamos saber la familias
# que tengan mayores a 3, con un True o False es suficiente

,family_size
0,False
1,False
2,False
3,False
4,False
...,...
886,False
887,False
888,False
889,False


In [24]:
total_large_family = large_family.sum()
total_large_family

np.int64(62)

In [26]:
# Proporción de supervivencia en las familias grandes
survival_rate_large_families = titanic.loc[large_family, 'survived'].mean()
survival_rate_large_families

np.float64(0.16129032258064516)

# Ejercicio 3: Segmentación por grupos de edad.

Clasifica a los pasajeros en las siguientes categorías de edad:

* menor de edad (< 18)
* mayor de edad (>=18)

Tip: Puede resultar mas sencillo realizarlo con una función

In [29]:
# Definir la función que recibe la edad y retorna si es mayor o menor
def age_group(age):
  if pd.isna(age):
    return 'Desconocido'
  elif age < 18:
    return 'Menor de edad'
  else:
    return 'Mayor de edad'

In [30]:
# Usar la función con apply
titanic['age_group'] = titanic['age'].apply(age_group)
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,age_group
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,Mayor de edad
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,1,Mayor de edad
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,0,Mayor de edad
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,1,Mayor de edad
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,0,Mayor de edad
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,0,Mayor de edad
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,0,Mayor de edad
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,3,Desconocido
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,0,Mayor de edad


In [31]:
# Esto sirve para validar lo anterior
titanic['age_group'].value_counts(dropna=False)


,count
age_group,
Mayor de edad,601
Desconocido,177
Menor de edad,113


# Ejercicio 4: Comparación entre promedios calculados manualmente y con Pandas

* Utiliza NumPy para calcular el promedio de las columnas 'Age' y 'Fare', ignorando valores nulos.
* Compara estos valores con los promedios obtenidos utilizando los métodos nativos de Pandas.
* Verifica que los resultados sean consistentes.

In [32]:
np.mean(titanic['age'])

np.float64(29.69911764705882)

In [36]:
np.mean(titanic['fare'])

np.float64(32.204207968574636)

In [37]:
titanic['age'].mean()

np.float64(29.69911764705882)

In [38]:
titanic['fare'].mean()

np.float64(32.204207968574636)

De lo anterior, usar nanmean y mean de numpy me entregó exactamente lo mismo, así que lo dejé con mean.

# Ejercicio 5. Creación de intervalos de clase usando NumPy y análisis con Pandas

* Divide la columna 'Fare' en 5 intervalos equidistantes utilizando la función numpy.linspace, el estudiante deberá investigar la operación de esta función en python.
* Crea una nueva columna en el DataFrame que asigne a cada pasajero el intervalo correspondiente de su tarifa.
* Calcula el número de pasajeros en cada intervalo utilizando Pandas y la proporción de supervivientes por intervalo.

### ¿Cómo funciona linspace?

Devuelve n números desde inicio al fin 'equiespaciados'

```python
np.linspace(inicio, fin, n)
```
Ejemplo

```python
np.linspace(0, 10, 5)
```

Resultado

```python
[0, 2.5, 5, 7.5, 10]
```


In [40]:
# Crear los intervalos, para eso obtener el mínimo y el máximo
fare_min = titanic['fare'].min()
fare_max = titanic['fare'].max()
fare_min, fare_max

(0.0, 512.3292)

In [41]:
# Hacer 6 cortes para 5 intervalos, también se les llama bins
fare_bins = np.linspace(fare_min, fare_max, 6)
fare_bins

array([  0.     , 102.46584, 204.93168, 307.39752, 409.86336, 512.3292 ])

In [43]:
# Usar cut para definir la columna categórica de cada pasajero
titanic['fare_category'] = pd.cut(titanic['fare'], bins=fare_bins, include_lowest=True)
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,age_group,fare_category
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,Mayor de edad,"(-0.001, 102.466]"
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,1,Mayor de edad,"(-0.001, 102.466]"
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,0,Mayor de edad,"(-0.001, 102.466]"
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,1,Mayor de edad,"(-0.001, 102.466]"
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,0,Mayor de edad,"(-0.001, 102.466]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,0,Mayor de edad,"(-0.001, 102.466]"
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,0,Mayor de edad,"(-0.001, 102.466]"
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,3,Desconocido,"(-0.001, 102.466]"
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,0,Mayor de edad,"(-0.001, 102.466]"


In [44]:
# Número de pasajeros por cada intervalo
passenger_count_by_interval = titanic['fare_category'].value_counts().sort_index()
passenger_count_by_interval

,count
fare_category,
"(-0.001, 102.466]",838
"(102.466, 204.932]",33
"(204.932, 307.398]",17
"(307.398, 409.863]",0
"(409.863, 512.329]",3


In [46]:
# Supervivencia por cada intervalo de fare
survival_rate_by_interval = titanic.groupby('fare_category', observed=True)['survived'].mean()
survival_rate_by_interval

,survived
fare_category,
"(-0.001, 102.466]",0.361575
"(102.466, 204.932]",0.757576
"(204.932, 307.398]",0.647059
"(409.863, 512.329]",1.000000
